In [11]:
SEED = 42

In [12]:
from pathlib import Path
from helpers.data.ticker_loader import load_tickers

TICKERS_FILE = Path("tickers.txt")
if not TICKERS_FILE.exists():
    TICKERS_FILE = Path("NN_Trading_project/tickers.txt")

# None        → all tickers
# ["AAPL", …] → explicit list
# 42          → random sample of 42 (seeded by SEED)
TICKER_SUBSET = 500

TICKERS = load_tickers(TICKERS_FILE, subset=TICKER_SUBSET, seed=SEED)
print(f"Using {len(TICKERS)} tickers")


Using 500 tickers


In [13]:
import pandas as pd
import datetime

# ============================================================
# DATE SETTINGS
# ============================================================

# --- Date Range ---
INTERVAL   = "1d"
START_DATE = pd.Timestamp("2025-01-01")           # inclusive
END_DATE   = pd.Timestamp(datetime.date.today())  # inclusive

# --- Train / Val / Test Split ---
# Train: Date <= TRAIN_END_DATE
# Val:   (TRAIN_END_DATE, VAL_END_DATE]
# Test:  Date > VAL_END_DATE
TRAIN_END_DATE = pd.Timestamp("2025-11-30")
VAL_END_DATE   = pd.Timestamp("2026-01-20")

# --- Exclusion Window (applied during featurize) ---
EXCLUDE_START_DATE = "2018-09-01"  # inclusive
EXCLUDE_END_DATE   = "2023-06-30"  # inclusive


# --- Exclusion Window (Features) ---
REBUILD_FEATURE_CACHE = True  # set True to recompute when features/WINDOW change


In [14]:
# ============================================================
# HYPERPARAMETERS - MODEL & TRAINING
# ============================================================

# --- Trading / Labeling ---
# Buy at market OPEN using prior day's OHLCV + current day's OPEN only.
HORIZON_BARS     = 0         # look-ahead bars for profit hit
PROFIT_THRESHOLD = 1 / 100   # 1% profit target vs current day's OPEN
STOP_LOSS        = -3 / 100  # -2% stop loss vs current day's OPEN
WINDOW           = 30        # look-back window for features

# --- Training ---
MAX_EPOCHS    = 50
PATIENCE      = 10   # early-stopping patience

BUY_THRESHOLD = 0.5  # minimum probability to classify as BUY

# --- Plotting ---
PLOT_GRAPH1_EPOCH_INTERVAL = 1
PLOT_GRAPH2_BATCH_INTERVAL = 5
PLOT_GRAPH3_EPOCH_INTERVAL = 1

# --- Optimizer ---
BATCH_SIZE   = 64

# --- Model Architecture ---
HIDDEN_SIZES = [64]  # e.g. [1024, 512] or [256, 128]

# --- Data / DataLoader ---
SPLIT_FRAC  = 0.85  # train fraction (time-based)
NUM_WORKERS = 16    # DataLoader workers (0 for debugging)


In [15]:
from pathlib import Path
from helpers.data.date_config_manager import check_and_refresh_date_config

_current_config = {
    "INTERVAL":           str(INTERVAL),
    "START_DATE":         str(START_DATE.date()),
    "END_DATE":           str(END_DATE.date()),
    "TRAIN_END_DATE":     str(TRAIN_END_DATE.date()),
    "VAL_END_DATE":       str(VAL_END_DATE.date()),
    "EXCLUDE_START_DATE": str(EXCLUDE_START_DATE),
    "EXCLUDE_END_DATE":   str(EXCLUDE_END_DATE),
    "TICKER_SUBSET":      str(TICKER_SUBSET),
}

check_and_refresh_date_config(
    current_config = _current_config,
    config_path    = Path.cwd() / "date_config.txt",
    stocks_dir     = Path.cwd() / "dataset" / "stocks",
)


Date config changed — clearing cached CSVs for a fresh download.
  Previous config:
    INTERVAL: 1d
    START_DATE: 2025-01-01
    END_DATE: 2026-04-02
    TRAIN_END_DATE: 2025-11-30
    VAL_END_DATE: 2026-01-20
    EXCLUDE_START_DATE: 2018-09-01
    EXCLUDE_END_DATE: 2023-06-30
    TICKER_SUBSET: 50 <-- CHANGED
  Deleted: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks
  Updated: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/date_config.txt


True

In [16]:
# Step 1: download NASDAQ data into the *dataset* directory using yfinance (DAILY)
from pathlib import Path
import pandas as pd
from helpers.data.data_downloader import download_tickers

# Root directory where yfinance CSVs will be stored

data_root = Path.cwd() / "dataset"
stocks_dir = data_root / "stocks"

# call helper
_summary = download_tickers(
    tickers= TICKERS,
    start=   START_DATE,
    end=     END_DATE,
    interval=INTERVAL,
    out_dir= stocks_dir,
)


 OK   :: AA rows=312 | downloaded 1/500
 OK   :: AAL rows=312 | downloaded 2/500
 OK   :: AAOI rows=312 | downloaded 3/500
 OK   :: AAPL rows=312 | downloaded 4/500
 OK   :: AB rows=312 | downloaded 5/500
 OK   :: ABEV rows=312 | downloaded 6/500
 OK   :: ABNB rows=312 | downloaded 7/500
 OK   :: ACI rows=312 | downloaded 8/500
 OK   :: ACIW rows=312 | downloaded 9/500
 OK   :: ADBE rows=312 | downloaded 10/500
 OK   :: ADI rows=312 | downloaded 11/500
 OK   :: ADP rows=312 | downloaded 12/500
 OK   :: ADSK rows=312 | downloaded 13/500
 OK   :: AEO rows=312 | downloaded 14/500
 OK   :: AEP rows=312 | downloaded 15/500
 OK   :: AES rows=312 | downloaded 16/500
 OK   :: AFRM rows=312 | downloaded 17/500
 OK   :: AGCO rows=312 | downloaded 18/500
 OK   :: AGNC rows=312 | downloaded 19/500
 OK   :: AJG rows=312 | downloaded 20/500
 OK   :: ALB rows=312 | downloaded 21/500


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ALE"}}}
$ALE: possibly delisted; no timezone found

1 Failed download:
['ALE']: possibly delisted; no timezone found


 OK   :: ALGN rows=312 | downloaded 22/500
 OK   :: ALKS rows=312 | downloaded 23/500
 OK   :: ALL rows=312 | downloaded 24/500
 OK   :: ALNY rows=312 | downloaded 25/500
 OK   :: AMAT rows=312 | downloaded 26/500
 OK   :: AMD rows=312 | downloaded 27/500
 OK   :: AMN rows=312 | downloaded 28/500
 OK   :: AMRZ rows=196 | downloaded 29/500
 OK   :: AMZN rows=312 | downloaded 30/500
 OK   :: ANET rows=312 | downloaded 31/500
 OK   :: AON rows=312 | downloaded 32/500
 OK   :: AOS rows=312 | downloaded 33/500
 OK   :: APA rows=312 | downloaded 34/500
 OK   :: APD rows=312 | downloaded 35/500
 OK   :: APGE rows=312 | downloaded 36/500
 OK   :: APP rows=312 | downloaded 37/500
 OK   :: AR rows=312 | downloaded 38/500
 OK   :: ARCB rows=312 | downloaded 39/500
 OK   :: ARE rows=312 | downloaded 40/500
 OK   :: ARWR rows=312 | downloaded 41/500
 OK   :: ASML rows=312 | downloaded 42/500
 OK   :: ASTE rows=312 | downloaded 43/500


$ATXS: possibly delisted; no price data found  (1d 2025-01-01 00:00:00 -> 2026-04-02 00:00:00) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ATXS']: possibly delisted; no price data found  (1d 2025-01-01 00:00:00 -> 2026-04-02 00:00:00) (Yahoo error = "No data found, symbol may be delisted")


 OK   :: AVA rows=312 | downloaded 44/500
 OK   :: AVAV rows=312 | downloaded 45/500
 OK   :: AVB rows=312 | downloaded 46/500
 OK   :: AVGO rows=312 | downloaded 47/500
 OK   :: AVTR rows=312 | downloaded 48/500
 OK   :: AXTA rows=312 | downloaded 49/500
 OK   :: BA rows=312 | downloaded 50/500
 OK   :: BABA rows=312 | downloaded 51/500
 OK   :: BAC rows=312 | downloaded 52/500
 OK   :: BANF rows=312 | downloaded 53/500
 OK   :: BAX rows=312 | downloaded 54/500
 OK   :: BAYRY rows=312 | downloaded 55/500
 OK   :: BBD rows=312 | downloaded 56/500
 OK   :: BBY rows=312 | downloaded 57/500
 OK   :: BFS rows=312 | downloaded 58/500
 OK   :: BHC rows=312 | downloaded 59/500
 OK   :: BHP rows=312 | downloaded 60/500
 OK   :: BIDU rows=312 | downloaded 61/500
 OK   :: BILI rows=312 | downloaded 62/500
 OK   :: BILL rows=312 | downloaded 63/500
 OK   :: BITF rows=312 | downloaded 64/500
 OK   :: BKH rows=312 | downloaded 65/500
 OK   :: BKNG rows=312 | downloaded 66/500
 OK   :: BKR rows=312 

...

 OK   :: LCID rows=312 | downloaded 262/500
 OK   :: LCTX rows=312 | downloaded 263/500
 OK   :: LEVI rows=312 | downloaded 264/500
 OK   :: LFUS rows=312 | downloaded 265/500
 OK   :: LHX rows=312 | downloaded 266/500
 OK   :: LI rows=312 | downloaded 267/500
 OK   :: LMND rows=312 | downloaded 268/500
 OK   :: LNT rows=312 | downloaded 269/500
 OK   :: LOGI rows=312 | downloaded 270/500
 OK   :: LOW rows=312 | downloaded 271/500
 OK   :: LSCC rows=312 | downloaded 272/500
 OK   :: LVS rows=312 | downloaded 273/500
 OK   :: MAA rows=312 | downloaded 274/500
 OK   :: MAC rows=312 | downloaded 275/500
 OK   :: MAN rows=312 | downloaded 276/500
 OK   :: MANH rows=312 | downloaded 277/500
 OK   :: MAR rows=312 | downloaded 278/500
 OK   :: MARA rows=312 | downloaded 279/500
 OK   :: MAS rows=312 | downloaded 280/500
 OK   :: MCHP rows=312 | downloaded 281/500
 OK   :: MDLZ rows=312 | downloaded 282/500
 OK   :: MELI rows=312 | downloaded 283/500
 OK   :: MGY rows=312 | downloaded 284/500



1 Failed download:
['NTRS']: TypeError("'NoneType' object is not subscriptable")


 OK   :: NVDA rows=312 | downloaded 314/500
 OK   :: NVS rows=312 | downloaded 315/500
 OK   :: NVTS rows=312 | downloaded 316/500
 OK   :: NWE rows=312 | downloaded 317/500
 OK   :: NXST rows=312 | downloaded 318/500
 OK   :: OCGN rows=312 | downloaded 319/500
 OK   :: OCUL rows=312 | downloaded 320/500
 OK   :: ODFL rows=312 | downloaded 321/500
 OK   :: OGE rows=312 | downloaded 322/500
 OK   :: OKE rows=312 | downloaded 323/500
 OK   :: OKTA rows=312 | downloaded 324/500
 OK   :: OLED rows=312 | downloaded 325/500
 OK   :: OMCL rows=312 | downloaded 326/500
 OK   :: ONDS rows=312 | downloaded 327/500
 OK   :: ONON rows=312 | downloaded 328/500
 OK   :: OPEN rows=312 | downloaded 329/500
 OK   :: OPY rows=312 | downloaded 330/500
 OK   :: ORGN rows=312 | downloaded 331/500
 OK   :: ORLY rows=312 | downloaded 332/500
 OK   :: OSPN rows=312 | downloaded 333/500
 OK   :: OSS rows=312 | downloaded 334/500
 OK   :: PANW rows=312 | downloaded 335/500
 OK   :: PCAR rows=312 | downloaded 33

$SPR: possibly delisted; no timezone found

1 Failed download:
['SPR']: possibly delisted; no timezone found


 OK   :: SPXC rows=312 | downloaded 405/500
 OK   :: SQM rows=312 | downloaded 406/500
 OK   :: SRAD rows=312 | downloaded 407/500
 OK   :: SRPT rows=312 | downloaded 408/500
 OK   :: SSB rows=312 | downloaded 409/500
 OK   :: SSNC rows=312 | downloaded 410/500
 OK   :: STE rows=312 | downloaded 411/500
 OK   :: STNE rows=312 | downloaded 412/500
 OK   :: STT rows=312 | downloaded 413/500
 OK   :: STWD rows=312 | downloaded 414/500
 OK   :: STZ rows=312 | downloaded 415/500
 OK   :: SUPN rows=312 | downloaded 416/500
 OK   :: SWBI rows=312 | downloaded 417/500
 OK   :: SXT rows=312 | downloaded 418/500
 OK   :: SYM rows=312 | downloaded 419/500
 OK   :: SYNA rows=312 | downloaded 420/500
 OK   :: SYY rows=312 | downloaded 421/500
 OK   :: TAK rows=312 | downloaded 422/500
 OK   :: TCBI rows=312 | downloaded 423/500
 OK   :: TCEHY rows=312 | downloaded 424/500
 OK   :: TCOM rows=312 | downloaded 425/500
 OK   :: TDW rows=312 | downloaded 426/500
 OK   :: TEAM rows=312 | downloaded 427/5

# Building Features + Labels

In [17]:
import pickle
import shutil
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from helpers.feature.feature_builder import precompute_and_cache, FEATURE_COLS
from helpers.data.dataset import StockDatasetSafe, is_cache_valid

# ── Paths ────────────────────────────────────────────────────────────────────
root       = Path.cwd() / "dataset"
stocks_dir = root / "stocks"
assert stocks_dir.exists(), f"Missing: {stocks_dir}"

files     = sorted(stocks_dir.glob("*.csv"))
cache_dir = Path.cwd() / f".feature_cache_forward_return_w{WINDOW}"
cache_dir.mkdir(parents=True, exist_ok=True)
scaler_path = cache_dir / "scaler.pkl"
index_path  = cache_dir / "index.pkl"

# ── Clear old cache if requested ─────────────────────────────────────────────
if REBUILD_FEATURE_CACHE:
    stale = [p for p in Path.cwd().glob(".feature*") if p.exists()]
    for p in stale:
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        else:
            p.unlink(missing_ok=True)
        print(f"[Cache] Removed: {p}")
    if not stale:
        print("[Cache] No stale .feature* paths found.")
    time.sleep(1)

# ── Build or load cache ───────────────────────────────────────────────────────
# Validates that scaler, index, AND every referenced .npz file all exist.
cache_ready = is_cache_valid(scaler_path, index_path)
if not cache_ready and cache_dir.exists():
    print("[Cache] Stale / incomplete cache detected — wiping cache dir.")
    shutil.rmtree(cache_dir)

# Ensure cache_dir exists before building or loading
cache_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_FEATURE_CACHE or not cache_ready:
    scaler, index = precompute_and_cache(
        files           = files,
        window          = WINDOW,
        cache_dir       = cache_dir,
        scaler_path     = scaler_path,
        index_path      = index_path,
        horizon_bars    = HORIZON_BARS,
        train_end_date  = TRAIN_END_DATE,
        val_end_date    = VAL_END_DATE,
        profit_threshold= PROFIT_THRESHOLD,
        stop_loss       = STOP_LOSS,
        exclude_start   = EXCLUDE_START_DATE,
        exclude_end     = EXCLUDE_END_DATE,
    )
else:
    print("[Cache] Using existing feature cache.")
    with open(scaler_path, "rb") as f: scaler = pickle.load(f)
    with open(index_path,  "rb") as f: index  = pickle.load(f)

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = StockDatasetSafe(index, scaler, "train")
val_ds   = StockDatasetSafe(index, scaler, "val")
test_ds  = StockDatasetSafe(index, scaler, "test")

# ── DataLoaders ───────────────────────────────────────────────────────────────
_pin    = torch.cuda.is_available()
_kwargs = dict(
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    pin_memory  = _pin,
    persistent_workers = (NUM_WORKERS > 0),
    prefetch_factor    = 2 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kwargs)

# ── Sanity check ──────────────────────────────────────────────────────────────
xb, yb = next(iter(train_loader))
print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")
print(f"Test  samples : {len(test_ds):,}")
print(f"X batch : {xb.shape}  {xb.dtype}")
print(f"y batch : {yb.shape}  {yb.dtype}")
print(f"Feature count : {len(FEATURE_COLS)} base + {WINDOW-1} lags = {len(FEATURE_COLS) * WINDOW}")

[Cache] Removed: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/.feature_cache_forward_return_w30
[Cache] Precomputing features (leakage-safe splits)...
[Cache] (1/495) AA.csv
[Cache] (2/495) AAL.csv
[Cache] (3/495) AAOI.csv
[Cache] (4/495) AAPL.csv
[Cache] (5/495) AB.csv
[Cache] (6/495) ABEV.csv
[Cache] (7/495) ABNB.csv
[Cache] (8/495) ACI.csv
[Cache] (9/495) ACIW.csv
[Cache] (10/495) ADBE.csv
[Cache] (11/495) ADI.csv
[Cache] (12/495) ADP.csv
[Cache] (13/495) ADSK.csv
[Cache] (14/495) AEO.csv
[Cache] (15/495) AEP.csv
[Cache] (16/495) AES.csv
[Cache] (17/495) AFRM.csv
[Cache] (18/495) AGCO.csv
[Cache] (19/495) AGNC.csv
[Cache] (20/495) AJG.csv
[Cache] (21/495) ALB.csv
[Cache] (22/495) ALGN.csv
[Cache] (23/495) ALKS.csv
[Cache] (24/495) ALL.csv
[Cache] (25/495) ALNY.csv
[Cache] (26/495) AMAT.csv
[Cache] (27/495) AMD.csv
[Cache] (28/495) AMN.csv
[Cache] (29/495) AMRZ.csv
[Cache] (30/495) AMZN.csv
[Cache] (31/495) ANET.csv
[Cache] (32/495) AON.csv
[Cache] (33/495) AOS.csv
[Cache

# XGBoost

In [18]:

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import log_loss

from helpers.evaluation import buy_metrics, predict_probs_booster

# ── Helpers ───────────────────────────────────────────────────────────────────

def loader_to_numpy(loader):
    """Flatten a torch DataLoader into (X, y) numpy arrays."""
    Xs, ys = [], []
    for xb, yb in loader:
        Xs.append(xb.numpy())
        ys.append(yb.numpy())
    return np.concatenate(Xs), np.concatenate(ys)


def logloss_curve(booster: xgb.Booster, X: np.ndarray, y: np.ndarray, iters: np.ndarray) -> np.ndarray:
    """Compute log-loss at each iteration count (post-training, for test curve)."""
    dmat = xgb.DMatrix(X)
    y = y.astype(np.int64)
    out = []
    for k in iters:
        try:
            p = booster.predict(dmat, iteration_range=(0, int(k)))
        except TypeError:
            p = booster.predict(dmat, ntree_limit=int(k))
        out.append(log_loss(y, p))
    return np.asarray(out, dtype=np.float64)


# ── 1. Build arrays from loaders ──────────────────────────────────────────────

X_train, y_train = loader_to_numpy(train_loader)
X_val,   y_val   = loader_to_numpy(val_loader)
X_test,  y_test  = loader_to_numpy(test_loader)

num_pos = float((y_train == 1).sum())
num_neg = float((y_train == 0).sum())
scale_pos_weight = num_neg / max(1.0, num_pos)

print(f"Train {X_train.shape}  pos={int(num_pos)}  neg={int(num_neg)}")
print(f"Val   {X_val.shape}    pos={int((y_val==1).sum())}  neg={int((y_val==0).sum())}")
print(f"Test  {X_test.shape}   pos={int((y_test==1).sum())}  neg={int((y_test==0).sum())}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

# ── 2. Define params ──────────────────────────────────────────────────────────

NUM_BOOST_ROUND       = 10000
EARLY_STOPPING_ROUNDS = max(5, PATIENCE * 5)

params = {
    "max_depth":        3,
    "eta":              0.01,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "alpha":            3.0,
    "lambda":           2.0,
    "scale_pos_weight": scale_pos_weight,
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "tree_method":      "hist",
}

# ── 3. Train ──────────────────────────────────────────────────────────────────

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

evals_result = {}
print(f"Training: max_rounds={NUM_BOOST_ROUND}, early_stop={EARLY_STOPPING_ROUNDS}")
booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    evals_result=evals_result,
    verbose_eval=100,
)

best_ntree = int(booster.best_iteration + 1) if booster.best_iteration is not None else NUM_BOOST_ROUND
print(f"\nBest iteration: {best_ntree}")

# ── 4. Final metrics ──────────────────────────────────────────────────────────

train_ll = np.asarray(evals_result["train"]["logloss"])
val_ll   = np.asarray(evals_result["val"]["logloss"])

probs_train = predict_probs_booster(booster, X_train, best_ntree)
probs_val   = predict_probs_booster(booster, X_val,   best_ntree)
probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
tr = buy_metrics(y_train, probs_train, BUY_THRESHOLD)
vl = buy_metrics(y_val,   probs_val,   BUY_THRESHOLD)
te = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)

print(f"Train  logloss={log_loss(y_train,probs_train):.6f}  acc={tr['acc']:.2f}%  P(success|BUY)={tr['buy_success']:.2f}%")
print(f"Val    logloss={log_loss(y_val,  probs_val  ):.6f}  acc={vl['acc']:.2f}%  P(success|BUY)={vl['buy_success']:.2f}%")
print(f"Test   logloss={log_loss(y_test, probs_test ):.6f}  acc={te['acc']:.2f}%  P(success|BUY)={te['buy_success']:.2f}%")

# ── 5. Training curves ────────────────────────────────────────────────────────

iters_full = np.arange(1, len(train_ll) + 1)

# ── 6. Save model ────────────────────────────────────────────────────────────

bundle = {"booster": booster, "best_ntree": best_ntree}
joblib.dump(bundle, "best_model_xgb.pkl")
print("Saved → best_model_xgb.pkl")


Train (60694, 1320)  pos=28086  neg=32608
Val   (16490, 1320)    pos=7763  neg=8727
Test  (24125, 1320)   pos=12809  neg=11316
scale_pos_weight = 1.1610
Training: max_rounds=10000, early_stop=50
[0]	train-logloss:0.69268	val-logloss:0.69268
[100]	train-logloss:0.67018	val-logloss:0.67266
[200]	train-logloss:0.66335	val-logloss:0.66921
[300]	train-logloss:0.65883	val-logloss:0.66824
[400]	train-logloss:0.65511	val-logloss:0.66798
[500]	train-logloss:0.65185	val-logloss:0.66769
[600]	train-logloss:0.64879	val-logloss:0.66748
[660]	train-logloss:0.64708	val-logloss:0.66753

Best iteration: 611
Train  logloss=0.648509  acc=61.51%  P(success|BUY)=56.87%
Val    logloss=0.667410  acc=58.34%  P(success|BUY)=54.78%
Test   logloss=0.678633  acc=56.95%  P(success|BUY)=56.65%
Saved → best_model_xgb.pkl


### Evaluate on Test Data 

In [19]:

from helpers.evaluation import evaluate_on_test_data

df_test_preds, df_test_summary = evaluate_on_test_data(
    booster=booster,
    ntree=best_ntree,
    X_test=X_test,
    y_test=y_test,
    threshold=BUY_THRESHOLD,
    index_path=index_path,
    stocks_dir=stocks_dir,
    save_csv="test_predictions_full.csv",
)


Saved test predictions -> test_predictions_full.csv
P(success | BUY): 56.65%  |  acc: 56.95%


# Optuna Hyperparameter Search (XGBoost)

In [ ]:

# ── Optuna: XGBoost Hyperparameter Search ─────────────────────────────────────

import numpy as np
import optuna
import xgboost as xgb
from sklearn.metrics import log_loss

optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_N_TRIALS     = 80
OPTUNA_EARLY_STOP   = max(10, PATIENCE * 5)
OPTUNA_MAX_ROUNDS   = 3000          # max boosting rounds per trial

dtrain_opt = xgb.DMatrix(X_train, label=y_train)
dval_opt   = xgb.DMatrix(X_val,   label=y_val)
dtest_opt  = xgb.DMatrix(X_test,  label=y_test)


def objective(trial: optuna.Trial) -> float:
    """Minimise validation log-loss."""
    params = {
        "objective":        "binary:logistic",
        "eval_metric":      "logloss",
        "tree_method":      "hist",
        "seed":             SEED,
        "scale_pos_weight": scale_pos_weight,
        # ── tuneable ────────────────────────────────────────────────────────
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "eta":              trial.suggest_float("eta", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "alpha":            trial.suggest_float("alpha", 0.0, 10.0),
        "lambda":           trial.suggest_float("lambda", 0.5, 10.0),
    }

    evals_res = {}
    bst = xgb.train(
        params=params,
        dtrain=dtrain_opt,
        num_boost_round=OPTUNA_MAX_ROUNDS,
        evals=[(dval_opt, "val")],
        early_stopping_rounds=OPTUNA_EARLY_STOP,
        evals_result=evals_res,
        verbose_eval=False,
    )

    best_iter   = int(bst.best_iteration + 1) if bst.best_iteration is not None else OPTUNA_MAX_ROUNDS
    val_logloss = evals_res["val"]["logloss"][best_iter - 1]

    trial.set_user_attr("best_ntree",   best_iter)
    trial.set_user_attr("val_logloss",  val_logloss)
    return val_logloss


# ── Run the study ─────────────────────────────────────────────────────────────

study = optuna.create_study(direction="minimize",
                             study_name="xgb_hparam_search",
                             sampler=optuna.samplers.TPESampler(seed=SEED))

print(f"Starting Optuna search: {OPTUNA_N_TRIALS} trials …")
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)

best_trial  = study.best_trial
best_params = best_trial.params
best_val_ll = best_trial.value
print(f"\nBest trial #{best_trial.number}  val_logloss={best_val_ll:.6f}")
print("Best params:", best_params)

# ── Retrain with best params on train+val ──────────────────────────────────────

final_params = {
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "tree_method":      "hist",
    "seed":             SEED,
    "scale_pos_weight": scale_pos_weight,
    **best_params,
}

# Use best_ntree from the winning trial (already tuned via early-stopping)
best_ntree_optuna = int(best_trial.user_attrs["best_ntree"])

dtrain_full = xgb.DMatrix(
    np.concatenate([X_train, X_val]),
    label=np.concatenate([y_train, y_val]),
)

print(f"\nRetraining on train+val for {best_ntree_optuna} rounds …")
booster = xgb.train(
    params=final_params,
    dtrain=dtrain_full,
    num_boost_round=best_ntree_optuna,
    verbose_eval=False,
)
best_ntree = best_ntree_optuna

# ── Evaluate on test ───────────────────────────────────────────────────────────

probs_test_optuna = predict_probs_booster(booster, X_test, best_ntree)
te_opt = buy_metrics(y_test, probs_test_optuna, BUY_THRESHOLD)
test_ll_val = log_loss(y_test, probs_test_optuna)

probs_train_optuna = predict_probs_booster(booster, X_train, best_ntree)
tr_opt = buy_metrics(y_train, probs_train_optuna, BUY_THRESHOLD)

print(f"\nTrain  acc={tr_opt['acc']:.2f}%  P(success|BUY)={tr_opt['buy_success']:.2f}%")
print(f"Test   acc={te_opt['acc']:.2f}%  P(success|BUY)={te_opt['buy_success']:.2f}%  logloss={test_ll_val:.6f}")

# ── Plots ──────────────────────────────────────────────────────────────────────

# 1. Optimisation history

# ── Save retrained model ──────────────────────────────────────────────────────
import joblib
bundle = {"booster": booster, "best_ntree": best_ntree}
joblib.dump(bundle, "best_model_xgb.pkl")
print("\nSaved → best_model_xgb.pkl")
print(f"Use BUY_THRESHOLD = {BUY_THRESHOLD} (unchanged; tune separately if needed)")


Starting Optuna search: 80 trials …


Best trial: 9. Best value: 0.666631:  12%|█▎        | 10/80 [16:10<2:11:41, 112.88s/it]

# Eval Data Analysis

In [ ]:

# ── Val Data Analysis: split-agnostic helper ──────────────────────────────────

import wandb
from helpers.evaluation import evaluate_split_from_artifacts

SELECTED_THRESHOLD = 0.9

val_preds, daily, val_summary = evaluate_split_from_artifacts(
    split="val",
    threshold=SELECTED_THRESHOLD,
    index_path=index_path,
    scaler_path=scaler_path,
    model_path="best_model_xgb.pkl",
    verbose=True,
)

display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "val_analysis/threshold":    SELECTED_THRESHOLD,
        "val_analysis/total_trades": int(val_summary["total_trades"]),
        "val_analysis/pct_success":  float(val_summary["pct_success"]),
        "val_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })


# Test Data Analysis

In [ ]:

# ── Test Data Analysis: split-agnostic helper ─────────────────────────────────

import wandb
from helpers.evaluation import evaluate_split_from_artifacts

SELECTED_THRESHOLD = 0.9

test_preds, daily, test_summary = evaluate_split_from_artifacts(
    split="test",
    threshold=SELECTED_THRESHOLD,
    index_path=index_path,
    scaler_path=scaler_path,
    model_path="best_model_xgb.pkl",
    verbose=True,
)

display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "test_analysis/threshold":    SELECTED_THRESHOLD,
        "test_analysis/total_trades": int(test_summary["total_trades"]),
        "test_analysis/pct_success":  float(test_summary["pct_success"]),
        "test_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })


In [ ]:
# don't execute the next cells
raise KeyboardInterrupt